# Welchen Kalender kennt das Stromnetz?

Aus dem kantonalen Stromverbrauch von Basel-Stadt (Viertelstundenwerte,
IWB über Open Data Basel-Stadt, Datensatz 100233) werden je Kalenderjahr
die zehn verbrauchsärmsten Tage ermittelt. Das Ergebnis landet in
`data/stillste-tage.json` und speist `index.html`.

Nur Standardbibliothek: `csv`, `datetime`, `statistics`, `json`, `collections`.
Die Analyse liest ausschliesslich die lokale Datei `data/verbrauch.csv`.

Regeln, die hier eingehalten werden:

* Datei aufsteigend sortieren (sie ist absteigend geliefert).
* `Wochentag` ist nullbasiert ab Montag; er wird aus dem Zeitstempel
  nachgerechnet, bei Abweichung bricht das Notebook ab.
* Tage werden nie über die Tagessumme verglichen, sondern über
  Mittelwert der Intervalle × 96 (Zeitumstellung: 92 oder 100 Intervalle).
* Tage mit anderer Intervallzahl als 92, 96, 100 sowie erster und letzter
  Tag der Datei werden verworfen.
* Rangfolge nur innerhalb eines Jahres, zusätzlich Wert in Prozent des
  Jahresmedians.
* Die Teilspalten `Grundversorgte Kunden` / `Freie Kunden` gelten erst ab
  1. September 2020 als vorhanden; früher werden sie als fehlend behandelt.
* Drei Zeitstempel sind laut Herausgeber interpoliert und werden markiert.

In [1]:
import csv
import datetime
import json
import statistics
from collections import defaultdict, Counter

CSV_PFAD = "data/verbrauch.csv"
JSON_PFAD = "data/stillste-tage.json"
ANZAHL_JE_JAHR = 10
GUELTIGE_INTERVALLZAHLEN = {92, 96, 100}
TEILSPALTEN_AB = datetime.date(2020, 9, 1)
INTERPOLIERT = {
    "2014-01-01T00:00:00+01:00",
    "2015-11-01T00:00:00+01:00",
    "2019-10-01T00:00:00+02:00",
}
ERWARTETE_KOPFZEILE = [
    "Start der Messung", "Start der Messung (Text)", "Stromverbrauch",
    "Grundversorgte Kunden", "Freie Kunden", "Jahr", "Monat", "Tag",
    "Wochentag", "Tag des Jahres", "Quartal", "Woche des Jahres",
]

## 1. Einlesen und aufsteigend sortieren

Die Datei beginnt mit einer BOM (`utf-8-sig` entfernt sie). Der Zeitstempel
trägt den Offset (`+01:00` / `+02:00`); `datetime.fromisoformat` versteht ihn.
Der Kalendertag eines Intervalls ist der lokale Tag im Zeitstempel.

In [2]:
zeilen = []
with open(CSV_PFAD, encoding="utf-8-sig", newline="") as f:
    leser = csv.DictReader(f, delimiter=";")
    assert leser.fieldnames == ERWARTETE_KOPFZEILE, leser.fieldnames
    for zeile in leser:
        zeilen.append(zeile)

for z in zeilen:
    z["ts"] = datetime.datetime.fromisoformat(z["Start der Messung"])
    z["wert"] = float(z["Stromverbrauch"])

zeilen.sort(key=lambda z: z["ts"])

print("Zeilen:", len(zeilen))
print("Erste Messung:", zeilen[0]["Start der Messung"])
print("Letzte Messung:", zeilen[-1]["Start der Messung"])

Zeilen: 515267
Erste Messung: 2012-01-01T00:15:00+01:00
Letzte Messung: 2026-09-11T23:45:00+02:00


## 2. Wochentag nachrechnen

Kontrolle der Konvention Montag = 0 gegen den Zeitstempel. Eine einzige
Abweichung bricht ab.

In [3]:
abweichungen = [
    z["Start der Messung"] for z in zeilen
    if z["ts"].weekday() != int(z["Wochentag"])
]
if abweichungen:
    raise SystemExit(
        f"Wochentag-Konvention verletzt bei {len(abweichungen)} Zeilen, "
        f"z. B. {abweichungen[:3]}"
    )
stichprobe = zeilen[-1]
print("Abweichungen:", len(abweichungen))
print("Stichprobe:", stichprobe["Start der Messung"],
      "weekday() =", stichprobe["ts"].weekday(),
      "Spalte =", stichprobe["Wochentag"])

Abweichungen: 0
Stichprobe: 2026-09-11T23:45:00+02:00 weekday() = 4 Spalte = 4


## 3. Intervalle je Kalendertag bündeln

Pro Tag: Anzahl Intervalle, Mittelwert × 96 als vergleichbarer Tageswert,
Markierung, falls ein interpolierter Zeitstempel enthalten ist. Für die
Teilspalten dasselbe, aber nur ab 1. September 2020 und nur, wenn beide
Felder in jedem Intervall des Tages gefüllt sind.

In [4]:
je_tag = defaultdict(list)
for z in zeilen:
    je_tag[z["ts"].date()].append(z)

tage = []
for datum in sorted(je_tag):
    intervalle = je_tag[datum]
    n = len(intervalle)
    gesamt = [z["wert"] for z in intervalle]
    eintrag = {
        "datum": datum,
        "intervalle": n,
        "wert": statistics.fmean(gesamt) * 96,
        "interpoliert": any(z["Start der Messung"] in INTERPOLIERT for z in intervalle),
        "grund": None,
        "frei": None,
    }
    if datum >= TEILSPALTEN_AB and all(
        z["Grundversorgte Kunden"] != "" and z["Freie Kunden"] != "" for z in intervalle
    ):
        eintrag["grund"] = statistics.fmean(float(z["Grundversorgte Kunden"]) for z in intervalle) * 96
        eintrag["frei"] = statistics.fmean(float(z["Freie Kunden"]) for z in intervalle) * 96
    tage.append(eintrag)

print("Kalendertage in der Datei:", len(tage))
print("Verteilung der Intervallzahlen:", dict(Counter(t["intervalle"] for t in tage)))

Kalendertage in der Datei: 5368
Verteilung der Intervallzahlen: {95: 1, 96: 5352, 92: 15}


## 4. Unvollständige Tage verwerfen

Erster und letzter Tag der Datei sind angeschnitten. Alle übrigen Tage mit
einer Intervallzahl ausserhalb von 92/96/100 fliegen ebenfalls raus.

Beobachtung: In diesem Stand hat kein Tag 100 Intervalle. Am Tag der
Umstellung auf Winterzeit fehlt die doppelte Stunde, der Tag hat 96
Intervalle. Die März-Tage haben wie erwartet 92.

In [5]:
erster, letzter = tage[0], tage[-1]
verworfen = [erster, letzter] + [
    t for t in tage[1:-1] if t["intervalle"] not in GUELTIGE_INTERVALLZAHLEN
]
gueltig = [t for t in tage[1:-1] if t["intervalle"] in GUELTIGE_INTERVALLZAHLEN]

for t in verworfen:
    print("verworfen:", t["datum"], "Intervalle:", t["intervalle"])
print("Gültige Tage:", len(gueltig))
assert all(t["intervalle"] in GUELTIGE_INTERVALLZAHLEN for t in gueltig)

verworfen: 2012-01-01 Intervalle: 95
verworfen: 2026-09-11 Intervalle: 96
Gültige Tage: 5366


## 5. Zehn stillste Tage je Jahr, relativ zum Jahresmedian

Der Median wird über alle gültigen Tage des Jahres gebildet. Das laufende
Jahr 2026 ist unvollständig (bis 10. September); sein Median und seine
Rangliste sind deshalb nur bedingt mit ganzen Jahren vergleichbar.

In [6]:
def rangliste(tage_liste, schluessel):
    """Zehn niedrigste Tage je Jahr nach `schluessel`, mit Prozent des Jahresmedians."""
    je_jahr = defaultdict(list)
    for t in tage_liste:
        if t[schluessel] is not None:
            je_jahr[t["datum"].year].append(t)
    ergebnis = []
    for jahr in sorted(je_jahr):
        liste = je_jahr[jahr]
        median = statistics.median(t[schluessel] for t in liste)
        stillste = sorted(liste, key=lambda t: t[schluessel])[:ANZAHL_JE_JAHR]
        ergebnis.append({
            "jahr": jahr,
            "tage": len(liste),
            "vollstaendig": len(liste) >= 360,
            "erster_tag": liste[0]["datum"].isoformat(),
            "letzter_tag": liste[-1]["datum"].isoformat(),
            "median": round(median, 1),
            "stillste": [
                {
                    "rang": i + 1,
                    "datum": t["datum"].isoformat(),
                    "wochentag": t["datum"].weekday(),
                    "wert": round(t[schluessel], 1),
                    "prozent": round(100 * t[schluessel] / median, 1),
                    "intervalle": t["intervalle"],
                    "interpoliert": t["interpoliert"],
                }
                for i, t in enumerate(stillste)
            ],
        })
    return ergebnis

gesamt_je_jahr = rangliste(gueltig, "wert")

WOCHENTAGE = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
for jahr in gesamt_je_jahr:
    print(f"\n{jahr['jahr']}  ({jahr['tage']} Tage, Median {jahr['median']:.0f})")
    for t in jahr["stillste"]:
        markierung = "  [interpolierter Zeitstempel]" if t["interpoliert"] else ""
        print(f"  {t['rang']:2d}. {t['datum']} {WOCHENTAGE[t['wochentag']]}"
              f"  {t['wert']:9.0f}  {t['prozent']:5.1f} %{markierung}")


2012  (365 Tage, Median 4518607)
   1. 2012-07-22 So    3246888   71.9 %
   2. 2012-04-08 So    3249198   71.9 %
   3. 2012-05-27 So    3285188   72.7 %
   4. 2012-12-30 So    3303635   73.1 %
   5. 2012-07-15 So    3310734   73.3 %
   6. 2012-05-13 So    3332182   73.7 %
   7. 2012-12-25 Di    3348907   74.1 %
   8. 2012-05-20 So    3353998   74.2 %
   9. 2012-09-16 So    3374334   74.7 %
  10. 2012-04-01 So    3394274   75.1 %

2013  (365 Tage, Median 4287867)
   1. 2013-05-19 So    3158376   73.7 %
   2. 2013-03-31 So    3158429   73.7 %
   3. 2013-12-29 So    3183470   74.2 %
   4. 2013-05-12 So    3186522   74.3 %
   5. 2013-06-30 So    3195782   74.5 %
   6. 2013-04-01 Mo    3228723   75.3 %
   7. 2013-05-05 So    3238943   75.5 %
   8. 2013-07-14 So    3246217   75.7 %
   9. 2013-04-14 So    3249068   75.8 %
  10. 2013-06-02 So    3251130   75.8 %

2014  (365 Tage, Median 4115746)
   1. 2014-04-20 So    2890324   70.2 %
   2. 2014-06-01 So    2999786   72.9 %
   3. 2014-05-04 S

## 6. Sind das die Feiertage?

Die Feiertage stehen nicht in den Daten. Für die Einordnung des Befunds
werden hier von Hand die gesetzlichen Feiertage von Basel-Stadt gebildet
(Neujahr, Karfreitag, Ostermontag, 1. Mai, Auffahrt, Pfingstmontag,
1. August, Weihnachten, Stephanstag), dazu das Fenster zwischen
Weihnachten und Neujahr (24.12.–2.1.) als «Betriebsferien». Diese Liste
dient nur der Kontrolle im Notebook; die Seite zeigt keine Feiertagsnamen.

In [7]:
def ostersonntag(jahr):
    """Gauss/Anonymous Gregorian algorithm."""
    a = jahr % 19
    b, c = divmod(jahr, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    monat, tag = divmod(h + l - 7 * m + 114, 31)
    return datetime.date(jahr, monat, tag + 1)

def feiertage_basel(jahr):
    ostern = ostersonntag(jahr)
    return {
        datetime.date(jahr, 1, 1): "Neujahr",
        ostern - datetime.timedelta(days=2): "Karfreitag",
        ostern + datetime.timedelta(days=1): "Ostermontag",
        datetime.date(jahr, 5, 1): "Tag der Arbeit",
        ostern + datetime.timedelta(days=39): "Auffahrt",
        ostern + datetime.timedelta(days=50): "Pfingstmontag",
        datetime.date(jahr, 8, 1): "Bundesfeiertag",
        datetime.date(jahr, 12, 25): "Weihnachten",
        datetime.date(jahr, 12, 26): "Stephanstag",
    }

def kategorie(datum):
    feiertage = feiertage_basel(datum.year)
    ostern = ostersonntag(datum.year)
    if datum == ostern:
        return "Ostersonntag"
    if datum == ostern + datetime.timedelta(days=49):
        return "Pfingstsonntag"
    if datum in feiertage:
        return "Feiertag (" + WOCHENTAGE[datum.weekday()] + ")"
    if datum.weekday() == 6:
        return "gewöhnlicher Sonntag"
    if datum.weekday() == 5:
        return "Samstag"
    if (datum.month == 12 and datum.day >= 24) or (datum.month == 1 and datum.day <= 2):
        return "Werktag zwischen Weihnachten und Neujahr"
    return "gewöhnlicher Werktag"

zaehler = Counter()
werktage = []
for jahr in gesamt_je_jahr:
    for t in jahr["stillste"]:
        d = datetime.date.fromisoformat(t["datum"])
        k = kategorie(d)
        zaehler[k] += 1
        if k.startswith("gewöhnlicher Werktag") or k.startswith("Werktag"):
            werktage.append((t["datum"], WOCHENTAGE[t["wochentag"]], t["prozent"]))

gesamt_anzahl = sum(zaehler.values())
sonntage = sum(1 for jahr in gesamt_je_jahr for t in jahr["stillste"] if t["wochentag"] == 6)
print(f"{gesamt_anzahl} Tage in den Ranglisten {gesamt_je_jahr[0]['jahr']}–{gesamt_je_jahr[-1]['jahr']}")
for k, n in zaehler.most_common():
    print(f"  {n:3d}  {k}")
print(f"\nDavon Sonntage (alle Kategorien): {sonntage} von {gesamt_anzahl}"
      f" = {100 * sonntage / gesamt_anzahl:.0f} %")
print("Werktage ausserhalb der Feiertage:", werktage if werktage else "keine")

print("\nStillster Tag je Jahr:")
rang1 = Counter()
for jahr in gesamt_je_jahr:
    t = jahr["stillste"][0]
    d = datetime.date.fromisoformat(t["datum"])
    k = kategorie(d)
    rang1[k] += 1
    print(f"  {jahr['jahr']}: {t['datum']} {WOCHENTAGE[t['wochentag']]}  {t['prozent']:.1f} %  {k}")
print("Rang 1 nach Kategorie:", dict(rang1))

print("\nWeihnachten (25.12.) und Neujahr (1.1.) in der Rangliste:")
for jahr in gesamt_je_jahr:
    treffer = [f"{t['datum']} Rang {t['rang']}" for t in jahr["stillste"]
               if t["datum"].endswith("-12-25") or t["datum"].endswith("-01-01")]
    print(f"  {jahr['jahr']}: {', '.join(treffer) if treffer else '–'}")

150 Tage in den Ranglisten 2012–2026
  101  gewöhnlicher Sonntag
   15  Ostersonntag
   11  Pfingstsonntag
    9  Feiertag (Mo)
    6  Feiertag (So)
    3  Feiertag (Fr)
    2  Feiertag (Sa)
    1  Feiertag (Di)
    1  Feiertag (Mi)
    1  Feiertag (Do)

Davon Sonntage (alle Kategorien): 133 von 150 = 89 %
Werktage ausserhalb der Feiertage: keine

Stillster Tag je Jahr:
  2012: 2012-07-22 So  71.9 %  gewöhnlicher Sonntag
  2013: 2013-05-19 So  73.7 %  Pfingstsonntag
  2014: 2014-04-20 So  70.2 %  Ostersonntag
  2015: 2015-12-27 So  72.3 %  gewöhnlicher Sonntag
  2016: 2016-03-27 So  72.5 %  Ostersonntag
  2017: 2017-04-16 So  74.0 %  Ostersonntag
  2018: 2018-04-01 So  74.6 %  Ostersonntag
  2019: 2019-04-21 So  75.6 %  Ostersonntag
  2020: 2020-04-12 So  77.0 %  Ostersonntag
  2021: 2021-05-23 So  74.7 %  Pfingstsonntag
  2022: 2022-04-17 So  74.1 %  Ostersonntag
  2023: 2023-04-09 So  75.9 %  Ostersonntag
  2024: 2024-03-31 So  75.3 %  Ostersonntag
  2025: 2025-08-03 So  78.2 %  gewö

## 7. Wie still sind Feiertage und Betriebsferien im Vergleich?

Die Rangliste allein sagt nicht, ob ein Feiertag auf einem Werktag «fast»
so still ist wie ein Sonntag. Deshalb hier der Median des relativen
Tageswerts je Kategorie über alle vollständigen Jahre.

In [8]:
je_kategorie = defaultdict(list)
jahresmedian = {j["jahr"]: j["median"] for j in gesamt_je_jahr}
for t in gueltig:
    d = t["datum"]
    if not gesamt_je_jahr[[j["jahr"] for j in gesamt_je_jahr].index(d.year)]["vollstaendig"]:
        continue
    k = kategorie(d)
    if k.startswith("Feiertag"):
        k = "Feiertag an einem Werktag" if d.weekday() < 5 else "Feiertag am Samstag"
    je_kategorie[k].append(100 * t["wert"] / jahresmedian[d.year])

print("Median des Tageswerts in Prozent des Jahresmedians, vollständige Jahre:")
for k, werte in sorted(je_kategorie.items(), key=lambda kv: statistics.median(kv[1])):
    print(f"  {statistics.median(werte):5.1f} %  {k} (n = {len(werte)})")

Median des Tageswerts in Prozent des Jahresmedians, vollständige Jahre:
   74.8 %  Ostersonntag (n = 14)
   76.9 %  Pfingstsonntag (n = 14)
   78.8 %  Feiertag am Samstag (n = 15)
   80.3 %  Feiertag an einem Werktag (n = 110)
   81.1 %  gewöhnlicher Sonntag (n = 694)
   88.1 %  Samstag (n = 723)
   92.2 %  Werktag zwischen Weihnachten und Neujahr (n = 69)
  102.2 %  gewöhnlicher Werktag (n = 3474)


## 8. Ausbaustufe: Grundversorgung und freier Markt getrennt (ab September 2020)

In [9]:
grund_je_jahr = rangliste(gueltig, "grund")
frei_je_jahr = rangliste(gueltig, "frei")

for name, liste in (("Grundversorgte Kunden", grund_je_jahr), ("Freie Kunden", frei_je_jahr)):
    print(f"\n=== {name} ===")
    for jahr in liste:
        print(f"{jahr['jahr']} ({jahr['tage']} Tage, Median {jahr['median']:.0f}):",
              ", ".join(f"{t['datum']} {WOCHENTAGE[t['wochentag']]} {t['prozent']:.0f}%"
                        for t in jahr["stillste"][:5]), "…")


=== Grundversorgte Kunden ===
2020 (122 Tage, Median 1654490): 2020-09-06 So 82%, 2020-09-20 So 82%, 2020-09-13 So 83%, 2020-10-04 So 85%, 2020-09-27 So 86% …
2021 (365 Tage, Median 1510763): 2021-07-11 So 79%, 2021-07-18 So 79%, 2021-08-08 So 80%, 2021-08-01 So 80%, 2021-06-13 So 82% …
2022 (365 Tage, Median 1456984): 2022-07-10 So 77%, 2022-07-31 So 79%, 2022-07-17 So 79%, 2022-04-17 So 80%, 2022-08-07 So 81% …
2023 (365 Tage, Median 1469617): 2023-07-30 So 77%, 2023-05-28 So 78%, 2023-07-23 So 79%, 2023-04-09 So 80%, 2023-07-02 So 81% …
2024 (366 Tage, Median 1539656): 2024-07-14 So 76%, 2024-07-13 Sa 79%, 2024-07-28 So 80%, 2024-07-07 So 81%, 2024-08-04 So 81% …
2025 (365 Tage, Median 1531691): 2025-08-03 So 76%, 2025-08-01 Fr 78%, 2025-07-13 So 79%, 2025-07-27 So 79%, 2025-08-24 So 79% …
2026 (253 Tage, Median 1513621): 2026-05-24 So 78%, 2026-06-07 So 80%, 2026-04-05 So 81%, 2026-06-14 So 81%, 2026-07-05 So 81% …

=== Freie Kunden ===
2020 (122 Tage, Median 1903756): 2020-12-20 

## 9. JSON schreiben

In [10]:
ausgabe = {
    "quelle": "Kantonaler Stromverbrauch, IWB über Open Data Basel-Stadt, Datensatz 100233, CC BY 4.0",
    "stand": zeilen[-1]["ts"].date().isoformat(),
    "einheit": "vermutlich kWh je 15-Minuten-Intervall, laut Portal prüfen",
    "methode": "Tageswert = Mittelwert der 15-Minuten-Intervalle × 96; Prozent = Tageswert / Median aller gültigen Tage des Jahres",
    "anzahl_je_jahr": ANZAHL_JE_JAHR,
    "verworfene_tage": [
        {"datum": t["datum"].isoformat(), "intervalle": t["intervalle"]} for t in verworfen
    ],
    "interpolierte_zeitstempel": sorted(INTERPOLIERT),
    "befund": {
        "ranglistenplaetze": gesamt_anzahl,
        "davon_sonntage": sonntage,
        "jahre": len(gesamt_je_jahr),
        "rang_1_nach_kategorie": dict(rang1),
        "kategorien": dict(zaehler),
        "median_prozent_je_kategorie_vollstaendige_jahre": {
            k: round(statistics.median(w), 1) for k, w in je_kategorie.items()
        },
    },
    "gesamt": gesamt_je_jahr,
    "segmente": {
        "grund": {"name": "Grundversorgte Kunden", "ab": TEILSPALTEN_AB.isoformat(), "jahre": grund_je_jahr},
        "frei": {"name": "Freie Kunden", "ab": TEILSPALTEN_AB.isoformat(), "jahre": frei_je_jahr},
    },
}

with open(JSON_PFAD, "w", encoding="utf-8") as f:
    json.dump(ausgabe, f, ensure_ascii=False, indent=1)

print("geschrieben:", JSON_PFAD)
print("Jahre gesamt:", [j["jahr"] for j in gesamt_je_jahr])

geschrieben: data/stillste-tage.json
Jahre gesamt: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


## 10. Seite speisen

`fetch()` auf eine lokale JSON-Datei wird von Browsern aus dem Dateisystem
blockiert. Damit `index.html` auch offline und ohne Server läuft, wird der
JSON-Inhalt zwischen Markierungen in die Seite eingebettet. Zusätzlich
wird die Liste aller Jahre als statisches HTML eingesetzt, damit die Seite
ohne JavaScript dieselben Tage mit sichtbarem Datum zeigt. Die Markierungen
bleiben stehen; der Rest der Seite wird nicht angefasst.

In [11]:
HTML_PFAD = "index.html"
WOCHENTAGE_LANG = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]
MONATE_LANG = ["Januar", "Februar", "März", "April", "Mai", "Juni", "Juli",
               "August", "September", "Oktober", "November", "Dezember"]

def zahl(n):
    return f"{round(n):,}".replace(",", "\u202f")

def datum_lang(iso, mit_wochentag=True):
    d = datetime.date.fromisoformat(iso)
    text = f"{d.day}. {MONATE_LANG[d.month - 1]} {d.year}"
    return f"{WOCHENTAGE_LANG[d.weekday()]}, {text}" if mit_wochentag else text

def html_jahr(jd, segment="gesamt", balken="var(--balken-b)"):
    teile = [f'<section class="jahr" id="jahr-{jd["jahr"]}" style="--balken: {balken}">',
             f'<h2>{jd["jahr"]}</h2>']
    info = f'{jd["tage"]} gültige Tage, Jahresmedian {zahl(jd["median"])}.'
    if not jd["vollstaendig"]:
        info += f' Unvollständiges Jahr, Daten bis {datum_lang(jd["letzter_tag"], False)}.'
    teile.append(f'<p class="info">{info}</p>')
    teile.append('<ol class="tage">')
    for t in jd["stillste"]:
        tid = f't-{segment}-{jd["jahr"]}-{t["rang"]}'
        teile.append('<li class="tag">')
        teile.append(f'<button type="button" aria-expanded="false" aria-controls="{tid}">')
        teile.append(f'<span class="rang">{t["rang"]}</span>')
        teile.append('<svg viewBox="0 0 100 8" preserveAspectRatio="none" aria-hidden="true">'
                     '<rect class="spur" x="0" y="0" width="100" height="8"/>'
                     f'<rect class="wert" x="0" y="0" width="{t["prozent"]}" height="8"/></svg>')
        teile.append(f'<span class="prozent">{zahl(t["prozent"])}\u202f%</span>')
        teile.append('<span class="hinweis">des Jahresmedians, Datum aufdecken</span>')
        teile.append('</button>')
        teile.append(f'<div class="datum" id="{tid}">')
        teile.append(f'<time datetime="{t["datum"]}">{datum_lang(t["datum"])}</time>')
        teile.append(f'<span class="absolut">Tageswert {zahl(t["wert"])}, {t["intervalle"]} Intervalle</span>')
        if t["interpoliert"]:
            teile.append('<span class="interpoliert">Enthält einen laut Herausgeber interpolierten Zeitstempel.</span>')
        teile.append('</div></li>')
    teile.append('</ol></section>')
    return "\n".join(teile)

def ersetzen(html, name, inhalt, block=True):
    anfang, ende = f"<!-- {name}-ANFANG -->", f"<!-- {name}-ENDE -->"
    a, e = html.index(anfang) + len(anfang), html.index(ende)
    trenner = "\n" if block else ""
    return html[:a] + trenner + inhalt + trenner + html[e:]

with open(JSON_PFAD, encoding="utf-8") as f:
    daten = json.load(f)

with open(HTML_PFAD, encoding="utf-8") as f:
    html = f.read()

html = ersetzen(html, "JAHRE", "\n".join(
    f'    <li><a href="#jahr-{j["jahr"]}">{j["jahr"]}</a></li>' for j in daten["gesamt"]))
html = ersetzen(html, "LISTE", "\n\n".join(html_jahr(j) for j in daten["gesamt"]))
html = ersetzen(html, "STAND", datum_lang(daten["stand"], False), block=False)
html = ersetzen(html, "DATEN", json.dumps(daten, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/"))

with open(HTML_PFAD, "w", encoding="utf-8") as f:
    f.write(html)

print("index.html aktualisiert:", len(html), "Zeichen,", len(daten["gesamt"]), "Jahre eingebettet")

index.html aktualisiert: 144637 Zeichen, 15 Jahre eingebettet
